# RACE: which neurons does a domain keep driving?

A layer's MLP writes the sum of its neurons' outputs into the residual stream, and so does the attention output projection. RACE scores each neuron on every token by how much of its write points along the module's whole update, then pools those scores over a set of texts. The ranking score is a lower credible bound on the neuron's mean, so a neuron that fires hard on a few tokens and idles on the rest ranks below one that pushes the same way on every token.

Method: Wang, Liu, Zhang, Han et al. (2026), "RACE: Scalable Statistical Estimation of Functional Consistency in LLM Neurons", EMNLP 2026 ([arXiv:2608.24758](https://arxiv.org/abs/2608.24758)). Reference code: [Nexround/RACE@cf6fb23](https://github.com/Nexround/RACE/tree/cf6fb23c22190c1c233f85256931e41441eda8a5) (MIT).

**Key questions**

- Which neurons consistently drive their module's update on indirect-object prompts?
- In which layers do the high scores sit?
- How many of the top neurons are generic, ranking high on plain sentiment sentences too?

**Structure**

1. Set up
2. Score two domains
3. Read the scores by layer
4. Filter out the reference domain
5. Save

**Model and data.** GPT-2 small in float32. The target domain is the twelve clean prompts of `murano.tasks.ioi`; the reference domain is the fifty sentences of `murano.tasks.sentiment`.

**Requirements.** `pip install murano-interp[plot]`

## 1. Set up

In [1]:
import torch

from murano import MuranoModel, Pipeline, keys
from murano.steps import RACE, LoadPrompts, Plot, Save
from murano.tasks import ioi, sentiment

OUTPUT_DIR = "murano_outputs/race"

# GPT-2 is small enough that bfloat16 rounding degrades its predictions.
model = MuranoModel("gpt2", dtype=torch.float32)

task = ioi(n=12)
positive, negative = sentiment()
reference_texts = positive + negative
print(task.clean[0])
print(reference_texts[0])

When Mary and John went to the store, John gave a drink to
I absolutely love this!


## 2. Score two domains

`RACE` runs each prompt once, teacher-forced, and scores every token. It keeps three numbers per neuron (count, mean and spread), so the prior and the credible level can change later without another forward pass.

In [2]:
results = Pipeline([LoadPrompts(task.clean), RACE(model)]).run()
target = results[keys.RACE]

reference = Pipeline([LoadPrompts(reference_texts), RACE(model)]).run()[keys.RACE]

print(f"IOI: {target.n} tokens, sentiment: {reference.n} tokens")
print(
    f"{len(target.mean)} modules scored:",
    ", ".join(str(n) for n in list(target.mean)[:4]),
    "...",
)

IOI: 168 tokens, sentiment: 328 tokens
24 modules scored: L0.mlp, L0.self_attn, L1.mlp, L1.self_attn ...


## 3. Read the scores by layer

In [3]:
Plot(output_dir=OUTPUT_DIR)(results);

Each ridge is one layer's distribution of positive CAM scores, on a log scale. Neurons scoring zero have no confident positive push and are left out. The largest score varies by almost two orders of magnitude between MLP layers (printed below), so selection ranks neurons within each layer and module instead of across the model.

In [4]:
cam = target.cam()
for node in sorted(cam):
    if node.module == "mlp":
        scores = cam[node]
        print(
            f"{str(node):<10} {int((scores > 0).sum()):>5} of {scores.numel()} > 0, max {scores.max():.2f}"
        )

L0.mlp       488 of 3072 > 0, max 3.32
L1.mlp       444 of 3072 > 0, max 8.91
L2.mlp       401 of 3072 > 0, max 36.25
L3.mlp       240 of 3072 > 0, max 4.34
L4.mlp       459 of 3072 > 0, max 1.13
L5.mlp       459 of 3072 > 0, max 0.55
L6.mlp       643 of 3072 > 0, max 1.19
L7.mlp       746 of 3072 > 0, max 2.00
L8.mlp       808 of 3072 > 0, max 5.59
L9.mlp       902 of 3072 > 0, max 9.39
L10.mlp     1056 of 3072 > 0, max 14.90
L11.mlp      874 of 3072 > 0, max 8.69


## 4. Filter out the reference domain

`select` keeps the top 1% of every layer and module. Passing `exclude=reference` walks the IOI ranking and skips the sentiment run's top 1%, so neurons that rank high on any English text drop out.

In [5]:
plain = target.select(fraction=0.01)
filtered = target.select(fraction=0.01, exclude=reference)
generic = set(reference.select(fraction=0.01))

print(f"{len(plain)} neurons selected per run")
shared = set(plain) & generic
by_module = {m: sum(n.node.module == m for n in shared) for m in ("mlp", "self_attn")}
print(f"shared with the sentiment top 1%, unfiltered: {len(shared)} {by_module}")
print(f"shared with the sentiment top 1%, filtered:   {len(set(filtered) & generic)}")
print("top IOI neurons after filtering:", [str(n) for n in filtered.neurons[:4]])

468 neurons selected per run
shared with the sentiment top 1%, unfiltered: 221 {'mlp': 167, 'self_attn': 54}
shared with the sentiment top 1%, filtered:   0
top IOI neurons after filtering: ['L0.mlp#1844', 'L0.mlp#623', 'L0.mlp#1329', 'L0.mlp#1596']


Nearly half of the unfiltered IOI selection, 221 of 468 neurons, also ranks in the sentiment top 1%: 167 MLP neurons and 54 attention neurons. The filtered selection keeps the same budget, filled from further down the IOI ranking.

## 5. Save

In [6]:
run_dir = Save(output_dir="murano_outputs", run_name="race", model_id=model.model_id)(
    results
)[keys.OUTPUT_DIR]
print("saved to:", run_dir)

saved to: murano_outputs/race


## What next

- [`ablation.ipynb`](ablation.ipynb) knocks out a component and measures the damage. The paper checks its neuron selections the same way, by zeroing them.
- [`logit_attribution.ipynb`](logit_attribution.ipynb) ranks whole heads and MLPs by their direct effect on the answer.